# Vector Stores in LangChain
[Chroma](https://www.trychroma.com/) (or ChromaDB) is `an open-source, AI-native vector database designed to store, manage, and search vector embeddings for large language model (LLM) applications`. It is known for being lightweight and developer-friendly, ideal for rapid prototyping and building applications that require understanding context and semantic meaning.

####  Key Features
- **Open-Source:** Available under the Apache 2.0 license, allowing for commercial use.
- **Vector Search:** Optimized for fast similarity searches using algorithms like Approximate Nearest Neighbor (ANN) and HNSW (Hierarchical Navigable Small World graphs), which helps find semantically related data points efficiently.
- **Integrated Functionality:** It provides an all-in-one solution that includes document storage, metadata filtering, and full-text search alongside vector search capabilities.

- **Language & Tool Integrations:** Offers client libraries for several languages, including Python, JavaScript, Ruby, Java, and Go. It integrates seamlessly with popular AI frameworks such as LangChain and LlamaIndex, as well as various embedding models from OpenAI, Google, and Hugging Face.
- **Deployment Flexibility:** Can be run locally on a laptop for development, deployed in a Docker container, or used as a managed service through Chroma Cloud for serverless, scalable production environments.
- **Multimodal Support:** Capable of handling various data types (text, images, etc.) by converting them into embeddings within the same vector space, enabling cross-modal searches (e.g., querying images with text).

#### Primary Use Cases

- **Retrieval-Augmented Generation (RAG):** Enhances AI chatbots and Q&A systems by providing them with access to proprietary or up-to-date information stored in the database, allowing them to generate more relevant and accurate responses.

- **Semantic Search:** Powers search engines that understand the intent and context of a user's query rather than relying on exact keywords.
- **Recommendation Systems:** Identifies items or content similar to a user's preferences, making it useful for e-commerce and media streaming applications.
- **Image and Video Retrieval:** Stores feature vectors for visual content, enabling the retrieval of similar images or video segments based on their visual characteristics.

In [51]:
!pip install langchain chromadb faiss-cpu tiktoken pypdf langchain-community -qU "langchain-chroma>=0.1.2" langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 27.9 MB/s eta 0:00:00


# Import Libraries

In [52]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

In [7]:
from langchain_core.documents import Document

# Download Embeddings Model

In [15]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},   # use "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True}
)


# Document Build

In [8]:
# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [18]:
docs = [doc1, doc2, doc3, doc4, doc5]


# Chroma DB

In [16]:
vector_store = Chroma(
    embedding_function = embedding_model,
    persist_directory = 'my_chroma_db',
    collection_name = 'ipl_data1'
)

## Add Data to Chroma

In [23]:
# vector_store.add_texts(texts = text)
vector_store.add_documents(documents = docs)

['52f385f2-4d6c-4b86-84f3-a68d6cac8b9d',
 '31e706f6-6d73-4b96-8781-4034a37eeab5',
 '202f166a-4ae6-4097-82ee-0098be89d404',
 'f1b544ab-8e23-456a-9941-4f6a2c1a8e92',
 'b6646e8e-d981-4988-8ce5-088c59ec563f']

## View Data

In [24]:
vector_store.get()

{'ids': ['52f385f2-4d6c-4b86-84f3-a68d6cac8b9d',
  '31e706f6-6d73-4b96-8781-4034a37eeab5',
  '202f166a-4ae6-4097-82ee-0098be89d404',
  'f1b544ab-8e23-456a-9941-4f6a2c1a8e92',
  'b6646e8e-d981-4988-8ce5-088c59ec563f'],
 'embeddings': None,
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.',
  'Ravindra Jadeja is a dynamic al

In [31]:
vector_store.get(
    ids=['52f385f2-4d6c-4b86-84f3-a68d6cac8b9d', '202f166a-4ae6-4097-82ee-0098be89d404'],
    include=['metadatas']
)

{'ids': ['52f385f2-4d6c-4b86-84f3-a68d6cac8b9d',
  '202f166a-4ae6-4097-82ee-0098be89d404'],
 'embeddings': None,
 'documents': None,
 'uris': None,
 'included': ['metadatas'],
 'data': None,
 'metadatas': [{'team': 'Royal Challengers Bangalore'},
  {'team': 'Chennai Super Kings'}]}

In [33]:
vector_store.get(
    ids=['202f166a-4ae6-4097-82ee-0098be89d404'],
    include=['metadatas', 'embeddings']
)

{'ids': ['202f166a-4ae6-4097-82ee-0098be89d404'],
 'embeddings': array([[-4.40126918e-02, -4.38647121e-02,  1.32785542e-02,
         -2.54441965e-02,  4.94078174e-02, -2.31735185e-02,
          3.65217240e-03, -1.52572319e-02, -4.22657803e-02,
         -6.38983473e-02,  1.04034960e-03, -1.49591612e-02,
         -2.57491469e-02,  4.45763655e-02,  7.68233314e-02,
          4.65372391e-02,  3.23287472e-02,  1.23738991e-02,
         -2.54768711e-02, -1.77486334e-02,  2.18533538e-02,
         -1.65425986e-02,  4.31004912e-02,  6.26874389e-03,
         -2.68449746e-02,  3.96270715e-02,  7.08254287e-03,
          2.58712377e-02, -2.52378788e-02, -3.19792777e-02,
          6.11843094e-02, -7.28451610e-02, -1.64238475e-02,
         -1.87714640e-02, -3.41864601e-02, -2.15242598e-02,
          1.21187102e-02,  2.69179940e-02,  7.74101494e-03,
         -3.89437787e-02, -2.68984511e-02, -3.70774902e-02,
          4.99682687e-02,  1.02811726e-03, -6.25220612e-02,
          1.98560022e-02,  3.2869104

## Search Documents

In [34]:
vector_store.similarity_search(
    query = 'who is captain?',
    k = 2
)

[Document(id='202f166a-4ae6-4097-82ee-0098be89d404', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
 Document(id='31e706f6-6d73-4b96-8781-4034a37eeab5', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [35]:
vector_store.similarity_search_with_score(
    query = 'who is captain?',
    k = 2
)

[(Document(id='202f166a-4ae6-4097-82ee-0098be89d404', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  0.7321245670318604),
 (Document(id='31e706f6-6d73-4b96-8781-4034a37eeab5', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  0.7351101040840149)]

## Meta Data Filtering

In [38]:
vector_store.similarity_search_with_score(
    query = '',
    k = 1,
    filter = {'team': 'Chennai Super Kings'}
)

[(Document(id='202f166a-4ae6-4097-82ee-0098be89d404', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.0925486087799072)]

## Update Document

In [39]:
update_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id = '52f385f2-4d6c-4b86-84f3-a68d6cac8b9d', document = update_doc1)

In [41]:
vector_store.get('52f385f2-4d6c-4b86-84f3-a68d6cac8b9d')

{'ids': ['52f385f2-4d6c-4b86-84f3-a68d6cac8b9d'],
 'embeddings': None,
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket."],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'team': 'Royal Challengers Bangalore'}]}

## Delete Document

In [42]:
vector_store.delete(ids = '52f385f2-4d6c-4b86-84f3-a68d6cac8b9d')
print('delete Done')

delete Done


In [43]:
vector_store.get(include = ['metadatas'])

{'ids': ['31e706f6-6d73-4b96-8781-4034a37eeab5',
  '202f166a-4ae6-4097-82ee-0098be89d404',
  'f1b544ab-8e23-456a-9941-4f6a2c1a8e92',
  'b6646e8e-d981-4988-8ce5-088c59ec563f'],
 'embeddings': None,
 'documents': None,
 'uris': None,
 'included': ['metadatas'],
 'data': None,
 'metadatas': [{'team': 'Mumbai Indians'},
  {'team': 'Chennai Super Kings'},
  {'team': 'Mumbai Indians'},
  {'team': 'Chennai Super Kings'}]}

# 2. FAISS

## Make Data

In [53]:
docs2 = [
    Document(
        page_content="Bangladesh has a strong agricultural economy, with rice being the main staple crop grown across the country.",
        metadata={"country": "Bangladesh", "sector": "Agriculture", "crop": "Rice"}
    ),
    Document(
        page_content="Agriculture employs a large portion of Bangladesh's population, especially in rural areas where farming is the primary livelihood.",
        metadata={"country": "Bangladesh", "sector": "Agriculture", "focus": "Employment"}
    ),
    Document(
        page_content="Bangladesh produces significant amounts of jute, often called the golden fiber, which is an important agricultural export.",
        metadata={"country": "Bangladesh", "sector": "Agriculture", "crop": "Jute"}
    ),
    Document(
        page_content="Fertile river delta soils and monsoon rains make Bangladesh suitable for growing crops such as rice, wheat, and vegetables.",
        metadata={"country": "Bangladesh", "sector": "Agriculture", "feature": "Soil and Climate"}
    ),
    Document(
        page_content="Modern agricultural techniques and irrigation systems are improving crop yields in Bangladesh despite climate challenges.",
        metadata={"country": "Bangladesh", "sector": "Agriculture", "topic": "Technology"}
    ),
]


## Add Documents Data to FAISS

In [54]:
faiss_db = FAISS.from_documents(
    documents = docs2,
    embedding = embedding_model
)

In [57]:
faiss_db.index_to_docstore_id  # dict mapping internal FAISS index -> your document IDs

{0: '375fe105-d91f-4e5c-bf1a-4fc18fc813a8',
 1: 'a9f0e19e-31a5-4a2f-b37f-b8d9448bdc55',
 2: '2dccf71c-2cc1-4279-b880-7c272b8516fa',
 3: '0b2f71be-cfe4-4821-9ce9-d44f54bbee49',
 4: '935d9062-10cc-46e0-8f69-cc61af54dcbd'}

## View Data

In [62]:
faiss_db.get_by_ids(['375fe105-d91f-4e5c-bf1a-4fc18fc813a8'])


[Document(id='375fe105-d91f-4e5c-bf1a-4fc18fc813a8', metadata={'country': 'Bangladesh', 'sector': 'Agriculture', 'crop': 'Rice'}, page_content='Bangladesh has a strong agricultural economy, with rice being the main staple crop grown across the country.')]

#### 3️⃣ Notes

* `docstore._dict` is the internal dictionary where FAISS stores all documents.
* `_dict` is “private” (hence the underscore), but it’s the easiest way to inspect all docs.
* For larger datasets, you might want to **page through the IDs** instead of loading everything at once.

In [65]:
# Print All Document
all_docs = list(faiss_db.docstore._dict.values())  # _dict stores all docs
for doc in all_docs:
    print('- ', doc.page_content)


-  Bangladesh has a strong agricultural economy, with rice being the main staple crop grown across the country.
-  Agriculture employs a large portion of Bangladesh's population, especially in rural areas where farming is the primary livelihood.
-  Bangladesh produces significant amounts of jute, often called the golden fiber, which is an important agricultural export.
-  Fertile river delta soils and monsoon rains make Bangladesh suitable for growing crops such as rice, wheat, and vegetables.
-  Modern agricultural techniques and irrigation systems are improving crop yields in Bangladesh despite climate challenges.


In [66]:
# filter by metadata
filtered_docs = [
    doc for doc in faiss_db.docstore._dict.values()
    if doc.metadata.get("crop") == "Rice"
]

for doc in filtered_docs:
    print(doc.page_content)


Bangladesh has a strong agricultural economy, with rice being the main staple crop grown across the country.


## Search Data

In [67]:
faiss_db.similarity_search(
    query = 'Rice',
    k = 2
)

[Document(id='375fe105-d91f-4e5c-bf1a-4fc18fc813a8', metadata={'country': 'Bangladesh', 'sector': 'Agriculture', 'crop': 'Rice'}, page_content='Bangladesh has a strong agricultural economy, with rice being the main staple crop grown across the country.'),
 Document(id='0b2f71be-cfe4-4821-9ce9-d44f54bbee49', metadata={'country': 'Bangladesh', 'sector': 'Agriculture', 'feature': 'Soil and Climate'}, page_content='Fertile river delta soils and monsoon rains make Bangladesh suitable for growing crops such as rice, wheat, and vegetables.')]

In [68]:
faiss_db.similarity_search_with_score(
    query = 'Rice',
    k = 2
)

[(Document(id='375fe105-d91f-4e5c-bf1a-4fc18fc813a8', metadata={'country': 'Bangladesh', 'sector': 'Agriculture', 'crop': 'Rice'}, page_content='Bangladesh has a strong agricultural economy, with rice being the main staple crop grown across the country.'),
  np.float32(0.6529623)),
 (Document(id='0b2f71be-cfe4-4821-9ce9-d44f54bbee49', metadata={'country': 'Bangladesh', 'sector': 'Agriculture', 'feature': 'Soil and Climate'}, page_content='Fertile river delta soils and monsoon rains make Bangladesh suitable for growing crops such as rice, wheat, and vegetables.'),
  np.float32(0.68334985))]

## Meta Data Filtering

In [74]:
faiss_db.similarity_search_with_score(
    query = '',
    k = 1,
    filter = {'feature': 'Soil and Climate'}
)

[(Document(id='0b2f71be-cfe4-4821-9ce9-d44f54bbee49', metadata={'country': 'Bangladesh', 'sector': 'Agriculture', 'feature': 'Soil and Climate'}, page_content='Fertile river delta soils and monsoon rains make Bangladesh suitable for growing crops such as rice, wheat, and vegetables.'),
  np.float32(1.1288986))]

# Update Document

In [76]:
faiss_db.get_by_ids(['375fe105-d91f-4e5c-bf1a-4fc18fc813a8'])

[Document(id='375fe105-d91f-4e5c-bf1a-4fc18fc813a8', metadata={'country': 'Bangladesh', 'sector': 'Agriculture', 'crop': 'Rice'}, page_content='Bangladesh has a strong agricultural economy, with rice being the main staple crop grown across the country.')]

In [75]:
# Data
new_doc = Document(
    page_content="Bangladesh is one of the world's largest producers of shrimp and fish, making aquaculture an important part of its agricultural sector.",
    metadata={"country": "Bangladesh", "sector": "Agriculture", "focus": "Aquaculture"}
)


### 1️⃣ Remove the old document

FAISS itself doesn’t directly “update” vectors by ID, but you can remove the old one from the docstore if you know its ID:

In [77]:
doc_id = "375fe105-d91f-4e5c-bf1a-4fc18fc813a8"

# Remove from docstore
if doc_id in faiss_db.docstore._dict:
    del faiss_db.docstore._dict[doc_id]


In [78]:
faiss_db.get_by_ids(['375fe105-d91f-4e5c-bf1a-4fc18fc813a8'])

[]

### 2️⃣ Add the new or updated document

In [79]:
faiss_db.add_documents([new_doc])

['e5a17909-1db3-44dd-a573-eb2109f62cf0']

In [80]:
faiss_db.get_by_ids(['e5a17909-1db3-44dd-a573-eb2109f62cf0'])

[Document(id='e5a17909-1db3-44dd-a573-eb2109f62cf0', metadata={'country': 'Bangladesh', 'sector': 'Agriculture', 'focus': 'Aquaculture'}, page_content="Bangladesh is one of the world's largest producers of shrimp and fish, making aquaculture an important part of its agricultural sector.")]